In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

In [3]:
cols = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent',
'hot','num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root',
'num_file_creations','num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login',
'count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
'diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count','dst_host_same_srv_rate',
'dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
'dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate',
'label','difficulty'
]

train_path = "/content/drive/MyDrive/AI_Lab/KDDTrain+.txt"
test_path  = "/content/drive/MyDrive/AI_Lab/KDDTest+.txt"

train_df = pd.read_csv(train_path, names=cols)
test_df  = pd.read_csv(test_path, names=cols)

In [4]:
cat_cols = ['protocol_type','service','flag']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])
    encoders[col] = le

In [5]:
train_df['label'] = (train_df['label'] != 'normal').astype(int)
test_df['label']  = (test_df['label'] != 'normal').astype(int)

In [6]:
num_cols = [c for c in cols if c not in cat_cols + ['label','difficulty']]

scaler = MinMaxScaler()
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
test_df[num_cols]  = scaler.transform(test_df[num_cols])

In [7]:
SEQ_LEN = 20

def create_sequences(df):
    data = df.drop(columns=['label','difficulty']).values
    labels = df['label'].values

    X, y = [], []
    for i in range(len(data) - SEQ_LEN):
        X.append(data[i:i+SEQ_LEN])
        y.append(labels[i+SEQ_LEN-1])  # classify last flow

    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_df)
X_test, y_test   = create_sequences(test_df)

In [8]:
class FlowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = FlowDataset(X_train, y_train)
test_dataset  = FlowDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32)

In [9]:
class NIDSTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=3):
        super().__init__()

        self.embedding = nn.Linear(input_dim, d_model)

        self.pos_embedding = nn.Parameter(torch.randn(1, 100, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.embedding(x)

        # positional encoding
        x = x + self.pos_embedding[:, :x.size(1), :]

        x = self.transformer(x)

        # last token
        x = x[:, -1, :]

        return self.classifier(x)

In [10]:
device = torch.device("cuda")

input_dim = X_train.shape[2]

model = NIDSTransformer(input_dim).to(device)

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0]).to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [11]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 0.1328
Epoch 2, Loss: 0.0852
Epoch 3, Loss: 0.0685
Epoch 4, Loss: 0.0580
Epoch 5, Loss: 0.0522
Epoch 6, Loss: 0.0475
Epoch 7, Loss: 0.0435
Epoch 8, Loss: 0.0398
Epoch 9, Loss: 0.0368
Epoch 10, Loss: 0.0348


In [12]:
from sklearn.metrics import classification_report

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)

        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = (probs > 0.4).int().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.71      0.97      0.82      9701
           1       0.97      0.70      0.81     12823

    accuracy                           0.81     22524
   macro avg       0.84      0.83      0.81     22524
weighted avg       0.85      0.81      0.81     22524

